In [0]:
%run /Users/che240008016@iiti.ac.in/01_translator 

In [0]:
%run /Users/che240008016@iiti.ac.in/03_context_injector

In [0]:
# ---- chatbot_main.py ----
class ExportAdvisorBot:
    def __init__(self):
        self.retriever    = load_retriever()
        self.chain        = build_rag_chain(self.retriever)
        self.user_lang    = "en-IN"
        self.product_code = None
        print("✅ Export Advisor Bot ready")

    def set_product(self, product_code):
        """Call this when user selects a product HS code"""
        self.product_code = product_code
        print(f"✅ Product set to: {product_code}")

    def chat(self, user_message):
        # Step 1 — Detect language and translate to English
        english_query, self.user_lang = translation_pipeline(user_message)

        # Step 2 — Enrich with opportunity scores if product is set
        if self.product_code:
            enriched_query, _ = inject_context_into_query(
                english_query,
                self.product_code
            )
        else:
            enriched_query = english_query

        # Step 3 — Run RAG chain
        result = self.chain({"question": enriched_query})
        answer = result["answer"]
        sources = list(set([
            doc.metadata.get("source", "unknown")
            for doc in result["source_documents"]
        ]))

        # Step 4 — Translate answer back to user language
        final_answer = translate_to_user_lang(answer, self.user_lang)

        return {
            "answer"       : final_answer,
            "sources"      : sources,
            "detected_lang": self.user_lang,
            "english_query": english_query
        }

bot = ExportAdvisorBot()

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True to VectorSearchClient().
✅ Retriever loaded
✅ RAG chain built
✅ Export Advisor Bot ready


✅ Context injector imports done


✅ Fetched 5 countries for product 120991
  Bahrain | Score: 1.7971 | Star Market
  Afghanistan | Score: 1.073 | Star Market
  China | Score: 0.9875 | Star Market
  Azerbaijan | Score: 0.966 | Star Market
  Bangladesh | Score: 0.9617 | Star Market


=== Export Opportunity Analysis ===


Rank 1: Bahrain (Asia)
- Product        : Seeds, fruit and spores, of a kind used for sowing : Other : Vegetable seeds
- Market Quadrant: Star Market
- Opportunity    : 1.7971 | Demand: 1.0454 | Risk: -1.5035
- Market Size    : $2.05M imports from 62 suppliers
- Tariff (AVE)   : 0.0
- Distance       : 2723.0 km from India
- Exchange Rate  : 1 USD = 220.0 INR
- Analysis       : Market: Emerging ($2.05M imports, 62 suppliers). Demand: Strong — low tariffs and close proximity. Risk: Very Low — strong currency and short distance. 
        

Rank 2: Afghanistan (Asia)
- Product        : Seeds, vegetable, nes for sowing
- Market Quadrant: Star Market
- Opportunity    : 1.073 | Demand: 0.9216 | Risk: -0.3028
- Market Size    : $14.96M imports from 63 suppliers
- Tariff (AVE)   : 0.02500000037252903
- Distance       : 1578.0 km from India
- Exchange Rate  : 1 USD = 1.0 INR
- Analysis       : Market: Medium ($15.0M imports, 63 suppliers). Demand: Strong — l

✅ Fetched 5 countries for product 120991

Why should I export to Bahrain?

Use this export opportunity data to support your answer:
=== Export Opportunity Analysis ===


Rank 1: Bahrain (Asia)
- Product        : Seeds, fruit and spores, of a kind used for sowing : Other : Vegetable seeds
- Market Quadrant: Star Market
- Opportunity    : 1.7971 | Demand: 1.0454 | Risk: -1.5035
- Market Size    : $2.05M imports from 62 suppliers
- Tariff (AVE)   : 0.0
- Distance       : 2723.0 km from India
- Exchange Rate  : 1 USD = 220.0 INR
- Analysis       : Market: Emerging ($2.05M imports, 62 suppliers). Demand: Strong — low tariffs and close proximity. Risk: Very Low — strong currency and short distance. 
        

Rank 2: Afghanistan (Asia)
- Product        : Seeds, vegetable, nes for sowing
- Market Quadrant: Star Market
- Opportunity    : 1.073 | Demand: 0.9216 | Risk: -0.3028
- Market Size    : $14.96M imports from 63 suppliers
- Tariff (AVE)   : 0.02500000037252903
- Distance       : 1578.0 k

In [0]:
# ---- set_product.py ----

# Set the product HS code from your opportunity scores output
bot.set_product("120991")

✅ Product set to: 120991


In [0]:
# ---- chat_loop.py ----
def chat_loop(bot):
    print("=" * 50)
    print("🇮🇳 Export Advisor Bot")
    print("Type 'exit' to quit")
    print("Type 'product:HSCODE' to change product")
    print("=" * 50)

    while True:
        user_input = input("\nYou: ").strip()

        if not user_input:
            continue

        if user_input.lower() == "exit":
            print("Goodbye!")
            break

        if user_input.lower().startswith("product:"):
            code = user_input.split(":")[1].strip()
            bot.set_product(code)
            print(f"✅ Switched to product {code}")
            continue

        response = bot.chat(user_input)

        print(f"\n🤖 Bot     : {response['answer']}")
        print(f"📄 Sources : {response['sources']}")
        print(f"🌐 Language: {response['detected_lang']}")

chat_loop(bot)

🇮🇳 Export Advisor Bot
Type 'exit' to quit
Type 'product:HSCODE' to change product



You:  product: 231101

✅ Product set to: 231101
✅ Switched to product 231101



You:  

In [0]:
# ---- test_chatbot.py ----

# Test English
r = bot.chat("What documents do I need to export seeds to Bahrain?")
print(f"Answer  : {r['answer']}")
print(f"Sources : {r['sources']}")
print(f"Lang    : {r['detected_lang']}")

print("\n" + "="*50)

# Test Hindi follow up
r = bot.chat("बहरीन के लिए यह बाजार क्यों अच्छा है?")
print(f"Answer  : {r['answer']}")
print(f"Sources : {r['sources']}")
print(f"Lang    : {r['detected_lang']}")

print("\n" + "="*50)

# Test Tamil
r = bot.chat("இந்த நாட்டிற்கு ஏற்றுமதி செய்வதற்கான ஆவணங்கள் என்ன?")
print(f"Answer  : {r['answer']}")
print(f"Sources : {r['sources']}")
print(f"Lang    : {r['detected_lang']}")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:512)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:632)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$5(UsageLogging.scala:659)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:147)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:349)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:59)
	at com.databricks.logging.AttributionContext$.withValue(Att